# Non-prior, Prior, Last-year 돌린 후 FAILED까지 수기처리 후 돌릴 것.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# [셀 2] POST-PROCESSING: 매핑 + long 변환 + currency 환율 적용
#        + channel: US 매핑 / dummy 0 / PAID_NONPAID
#        + metric_col 키 분리 (J1~J7, J11)
#        + 개별 _stacked_separate.csv + 전체 union 저장
# ─────────────────────────────────────────────────────────────────
import re
from pathlib import Path
from datetime import datetime
import pandas as pd

MAPPING_CSV  = Path.cwd() / "tb_column_name_mapping.csv"
CURRENCY_CSV = Path.cwd() / "currency.csv"
EXPORTS_DIR  = Path.cwd() / "aa_exports"

# 리포트 번호 패턴 (1_1, 3_2 등): 이 패턴이 있는 파일만 dummy 0 삽입 / CSV 저장 / union 포함
_REPORT_NO_PAT = re.compile(r'_\d{1,2}_\d{1,2}(?:_|$)')

# ── currency 로드 ─────────────────────────────────────────────────
cur_df = pd.read_csv(CURRENCY_CSV)
col_latest = cur_df.columns[2]
col_prior  = cur_df.columns[3]
year_latest = str(pd.to_datetime(col_latest).year)
year_prior  = str(pd.to_datetime(col_prior).year)
print(f"▶ 환율 컬럼: {col_latest}({year_latest}) / {col_prior}({year_prior})")

cur_map = {
    str(r["site_code"]).strip().lower(): {
        year_latest: r[col_latest],
        year_prior:  r[col_prior],
    }
    for _, r in cur_df.iterrows()
}

# ── channel 매핑 정의 ─────────────────────────────────────────────
us_mapping = {
    "US Channel":"Global Channel Mapping ",
    "Affiliate":"Affiliate Marketing",
    "Display":"Display",
    "Display Retargeting":"Display",
    "Other External Campaign":"Paid Others",
    "Other Paid Ecomm":"Paid Others",
    "Paid Search":"Paid Search",
    "Paid Search - eComm":"Paid Search",
    "PLA":"Pmax",
    "Social (Paid)":"Social Media Campaigns",
    "Vanity":"Paid Others",
    "Direct":"Direct",
    "Email - CRM":"Email",
    "Email - eComm":"Email",
    "Email - Upsell it":"Email",
    "Email (Retired)":"Email",
    "EPP":"EPP - US",
    "Natural Search":"Natural Search",
    "Other":"Other",
    "None":"Other",
    "Push Notifications":"Push",
    "Referring Domains":"Other",
    "Session Refresh":"Session Refresh",
    "Social (Free and Owned)":"Owned Social",
    "Social (Retired)":"Social Network Referrals",
    "Other External CampaignSegments":"Mobile Application",
    "Other External CampaignUS_Smartthings":"Mobile Application - Smartthings",
    "Other External CampaignUS_Samsung Members":"Mobile Application - Samsung Members",
    "SMS":"SMS",
    "Gen AI Search":"Gen AI (Organic)"
}

global_paid_mapping = {
    "Paid Search":"Paid",
    "Social Media Campaigns":"Paid",
    "Affiliate Marketing":"Paid",
    "Display":"Paid",
    "Other Campaigns":"Paid",
    "QR code (Paid)":"Paid",
    "Pmax":"Paid",
    "Demand Gen":"Paid",
    "Paid Others":"Paid",
    "Video":"Paid",
    "Gen AI (Paid)":"Paid",
    "Session Refresh":"Non-Paid",
    "Email":"Non-Paid",
    "Direct":"Non-Paid",
    "Natural Search":"Non-Paid",
    "Mobile Application":"Non-Paid",
    "Push":"Non-Paid",
    "Social Network Referrals":"Non-Paid",
    "Other":"Non-Paid",
    "SMS":"Non-Paid",
    "Owned Social":"Non-Paid",
    "QR code (Owned)":"Non-Paid",
    "Samsung Web":"Non-Paid",
    "Gen AI (Organic)":"Non-Paid",
    "Owned Others":"Non-Paid"
}
# ※ "None" 채널은 "Other"로 통일하므로 global_paid_mapping에서 제외

# ── 최종 컬럼 정리 ────────────────────────────────────────────────
def finalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "PAID_NONPAID" not in df.columns:
        df["PAID_NONPAID"] = "-"
    df["TIER"]  = ""
    df["공란1"] = ""
    df["공란2"] = ""
    df["공란3"] = ""
    df["공란4"] = ""
    if "Site_Code" in df.columns and "metric_col" in df.columns:
        df["metric_col"] = df["Site_Code"].str.strip().str.lower() + "_" + df["metric_col"].astype(str)
    df = df.rename(columns={
        "Subsidiary":         "SUBS",
        "Country":            "COUNTRY",
        "Site_Code":          "SITE CODE",
        "J11":                "REPORT NO.",
        "J3":                 "DIVISION",
        "J4":                 "DATE",
        "J5":                 "DEVICE TYPE",
        "J6":                 "TYPE",
        "J7":                 "LOGIN/NON",
        "PAID_NONPAID":       "PAID/NONPAID",
        "value":              "ITEM",
        "metric_value_adj":   "VALUE",
        "metric_col":         "KEY",
        "metric_value_origin":"value_origin",
        "Start_Date":         "start_date",
        "End_Date":           "end_date",
    })
    final_cols = [
        "TIER", "SUBS", "COUNTRY", "SITE CODE",
        "REPORT NO.", "DIVISION", "DATE", "DEVICE TYPE", "TYPE", "LOGIN/NON",
        "PAID/NONPAID", "ITEM", "VALUE", "KEY",
        "공란1", "공란2", "공란3", "공란4",
        "value_origin", "start_date", "end_date"
    ]
    return df[[c for c in final_cols if c in df.columns]]


# ── 캠페인 키 분리 유틸 ───────────────────────────────────────────
report_no_mapping = {
    "1_1": "1_1~2. S.com Traffic by Division",
    "2_1": "2_1~4. Basic Traffic",
    "3_1": "3_1. Traffic by Channel (Internal)",
    "3_2": "3_2. Traffic by Channel (External)",
    "3_3": "3_3. Home KV & GNB to Campaign Page",
    "4_1": "4_1. Order Conversion with Login/Non_Login",
    "4_2": "4_2. Order Conversion with Login/Non_Login (Visit)",
    "5_1": "5_1. S.com Order Conversion",
    "6_1": "6_1. S.com Cross Sell Order (Multi Purchase)",
    "6_2": "6_2. S.com Cross Sell Order (Total)",
    "6_3": "6_3. Campaign Page Cross Sell Order",
    "7_1": "7_1~2. Order Conversion/Traffic by Channel"
}

def _split_by_number_pattern(parts):
    if len(parts) >= 2 and re.fullmatch(r"\d{1,2}", parts[0]) and re.fullmatch(r"\d{1,2}", parts[1]):
        return "", f"{parts[0]}_{parts[1]}", parts[2:]
    for i in (2, 3):
        if i + 1 < len(parts) and parts[i].isdigit() and parts[i+1].isdigit():
            return '_'.join(parts[:i]), f"{parts[i]}_{parts[i+1]}", parts[i+2:]
    return parts[0], f"{parts[1]}_{parts[2]}", parts[3:]

def _find_report_key(values):
    for v in values:
        if isinstance(v, str) and re.fullmatch(r"\d+_\d+", v):
            return v
    return ''

def split_metric_col(val: str) -> dict:
    parts = str(val).split('_')
    if len(parts) < 3:
        j1_to_j7 = (parts + [''] * 7)[:7]
        return {**{f"J{i+1}": j1_to_j7[i] for i in range(7)}, "J11": "", "metric_name": ""}
    first, second, remaining = _split_by_number_pattern(parts)
    processed = []
    i = 0
    while i < len(remaining):
        if remaining[i].isdigit() and remaining[i].startswith('202'):
            if i + 2 < len(remaining) and remaining[i+2].lower() == 'prior':
                processed.append(f"{remaining[i]}_{remaining[i+1]}_{remaining[i+2]}")
                i += 3
            elif i + 1 < len(remaining):
                processed.append(f"{remaining[i]}_{remaining[i+1]}")
                i += 2
            else:
                processed.append(remaining[i])
                i += 1
        else:
            processed.append(remaining[i])
            i += 1
    all_parts = [first, second] + processed
    j1_to_j7 = (all_parts + [''] * 7)[:7]
    metric_name = '_'.join(all_parts[7:]) if len(all_parts) > 7 else ''
    report_key = _find_report_key(j1_to_j7)
    return {**{f"J{i+1}": j1_to_j7[i] for i in range(7)}, "J11": report_no_mapping.get(report_key, ''), "metric_name": metric_name}

def _apply_j_cols(df: pd.DataFrame) -> pd.DataFrame:
    records = [split_metric_col(v) for v in df["metric_col"]]
    if not records:
        extra_cols = [f"J{i+1}" for i in range(7)] + ["J11", "metric_name"]
        df = df.copy()
        for col in extra_cols:
            df[col] = pd.NA
        return df
    j_df = pd.DataFrame(records, index=df.index)
    return pd.concat([df, j_df], axis=1)

# ── non-channel dummy 0 삽입 유틸 ────────────────────────────────
def _insert_nonchannel_dummy(df_long: pd.DataFrame, tb_key: str,
                              master_sites: set, site_meta_map: dict) -> pd.DataFrame:
    """마스터 site_code 기준으로 누락된 site×metric_col 조합에 0행 삽입"""
    all_sites   = list(master_sites)
    metric_cols = df_long["metric_col"].unique()

    meta_cols = [c for c in ["Subsidiary", "Country", "RSID"] if c in df_long.columns]
    file_meta = (
        df_long.groupby("Site_Code")[meta_cols]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        .reset_index()
    )
    date_cols = [c for c in ["Start_Date", "End_Date", "itemId"] if c in df_long.columns]
    file_date_meta = (
        df_long.groupby("Site_Code")[date_cols]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        .reset_index()
    )
    file_date_mode = {
        c: df_long[c].mode().iloc[0] if not df_long[c].mode().empty else None
        for c in date_cols
    }

    cross = pd.MultiIndex.from_product(
        [all_sites, metric_cols],
        names=["Site_Code", "metric_col"]
    ).to_frame(index=False)

    existing = df_long[["Site_Code", "metric_col"]].drop_duplicates()
    existing["_exists"] = True

    merged = cross.merge(existing, on=["Site_Code", "metric_col"], how="left")
    dummy_rows = merged[merged["_exists"].isna()].drop(columns=["_exists"])

    if dummy_rows.empty:
        return df_long

    dummy_rows = dummy_rows.merge(file_meta, on="Site_Code", how="left")
    dummy_rows = dummy_rows.merge(file_date_meta, on="Site_Code", how="left")

    for _col in meta_cols:
        if _col not in dummy_rows.columns:
            continue
        _na_mask = dummy_rows[_col].isna()
        if _na_mask.any():
            dummy_rows.loc[_na_mask, _col] = dummy_rows.loc[_na_mask, "Site_Code"].map(
                lambda s: site_meta_map.get(s, {}).get(_col)
            )

    for _col, _val in file_date_mode.items():
        if _col in dummy_rows.columns:
            dummy_rows[_col] = dummy_rows[_col].fillna(_val)

    dummy_rows["metric_value_origin"] = 0.0
    dummy_rows["metric_value_adj"]    = 0.0
    dummy_rows = _apply_j_cols(dummy_rows)
    dummy_rows["value"] = dummy_rows["metric_name"]
    df_long = pd.concat([df_long, dummy_rows], ignore_index=True)
    print(f"  {tb_key} → dummy {len(dummy_rows)}행 삽입")
    return df_long

# ── 매핑 로드 ─────────────────────────────────────────────────────
mapping_df = pd.read_csv(MAPPING_CSV)
id_vars = ["Subsidiary", "Country", "Site_Code", "RSID",
           "Start_Date", "End_Date", "itemId", "value"]

ok_list, skip_list, no_vars_list, no_report_list = [], [], [], []
channel_frames = {}
all_frames = []

# ── [260316 개선] tb_key별 최신 파일만 선택 ──────────────────────
_all_csv = [f for f in EXPORTS_DIR.glob("*.csv")
            if "_long" not in f.name
            and "_stacked" not in f.name
            and not f.name.startswith("union_")]
_TS_PAT = re.compile(r"_(\d{8}_\d{6})$")

def _get_ts(p):
    m = _TS_PAT.search(p.stem)
    return m.group(1) if m else ""

_latest_map = {}
for _f in _all_csv:
    _key = _TS_PAT.sub("", _f.stem)
    if _key not in _latest_map or _get_ts(_f) > _get_ts(_latest_map[_key]):
        _latest_map[_key] = _f

csv_files = sorted(_latest_map.values())
print(f"▶ 처리 대상: {len(csv_files)}개 파일 (tb_key별 최신 파일 기준)\n")

# ── Pre-scan: 전체 site_code 및 메타 정보 수집 ───────────────────
_META_COLS = ["Subsidiary", "Country", "RSID"]
_global_sites = set()
_us_sites     = set()
_site_meta_map = {}  # site_code → {Subsidiary, Country, RSID}

for _path in csv_files:
    _tk = re.sub(r"_\d{8}_\d{6}$", "", _path.stem)
    try:
        _usecols = lambda c: c in (["Site_Code"] + _META_COLS)
        _tmp = pd.read_csv(_path, encoding="utf-8-sig", usecols=_usecols)
        for _sc_raw in _tmp["Site_Code"].dropna().unique():
            _sc = str(_sc_raw).strip()
            if _sc not in _site_meta_map:
                _row = _tmp[_tmp["Site_Code"].astype(str).str.strip() == _sc].iloc[0]
                _site_meta_map[_sc] = {c: _row[c] for c in _META_COLS if c in _tmp.columns}
            if _tk.startswith("us_"):
                _us_sites.add(_sc)
            else:
                _global_sites.add(_sc)
    except Exception:
        pass

print(f"▶ Pre-scan 완료")
print(f"  Global site_codes ({len(_global_sites)}개): {sorted(_global_sites)}")
print(f"  US site_codes ({len(_us_sites)}개): {sorted(_us_sites)}\n")

# ── 메인 처리 루프 ────────────────────────────────────────────────
for src_path in sorted(csv_files):
    stem   = src_path.stem
    tb_key = re.sub(r"_\d{8}_\d{6}$", "", stem)

    # 리포트 번호 없는 파일은 전체 skip (bestselling, nextpage, raw_multi_purchase_v41 등)
    if not _REPORT_NO_PAT.search(tb_key) and "channel" not in tb_key.lower():
        no_report_list.append(tb_key)
        continue

    tb_mapping = mapping_df[mapping_df["tb"] == tb_key]
    if tb_mapping.empty:
        tb_mapping = mapping_df[mapping_df["tb"] == re.sub(r"_prior$", "", tb_key)]

    if tb_mapping.empty:
        skip_list.append(tb_key)
        continue

    rename_dict = dict(zip(tb_mapping["value_n"], tb_mapping["column"]))

    df = pd.read_csv(src_path, encoding="utf-8-sig")
    df = df.rename(columns=rename_dict)
    value_vars = [c for c in rename_dict.values() if c in df.columns]

    if not value_vars:
        no_vars_list.append(tb_key)
        continue

    for v in value_vars:
        df[v] = pd.to_numeric(
            df[v].astype(str).str.strip().str.replace("'", "", regex=False),
            errors="coerce"
        )

    df_long = df.melt(
        id_vars=[c for c in id_vars if c in df.columns],
        value_vars=value_vars,
        var_name="metric_col",
        value_name="metric_value_origin"
    )

    # ── currency 환율 적용 ────────────────────────────────────────
    df_long["_year"] = pd.to_datetime(df_long["Start_Date"]).dt.year.astype(str)
    df_long["_site"] = df_long["Site_Code"].str.strip().str.lower()

    def calc_adj(row):
        if "revenue" not in str(row["metric_col"]).lower():
            return row["metric_value_origin"]
        rates = cur_map.get(row["_site"])
        if not rates:
            return row["metric_value_origin"]
        rate = rates.get(row["_year"])
        if rate is None:
            return row["metric_value_origin"]
        try:
            return float(row["metric_value_origin"]) * float(rate)
        except (TypeError, ValueError):
            return row["metric_value_origin"]

    df_long["metric_value_adj"] = df_long.apply(calc_adj, axis=1)
    df_long = df_long.drop(columns=["_year", "_site"])

    df_long["metric_value_origin"] = pd.to_numeric(
        df_long["metric_value_origin"].astype(str).str.replace("'", "", regex=False),
        errors="coerce"
    )
    df_long["metric_value_adj"] = pd.to_numeric(
        df_long["metric_value_adj"].astype(str).str.replace("'", "", regex=False),
        errors="coerce"
    )

    # ── metric_col 키 분리 ────────────────────────────────────────
    df_long = _apply_j_cols(df_long)

    # ── channel 파일이면 수집, 아니면 dummy 0 삽입 후 저장 ────────
    is_channel = "channel" in tb_key.lower()
    if is_channel:
        channel_frames[tb_key] = df_long
    else:
        df_long["value"] = df_long["metric_name"]
        master = _us_sites if tb_key.startswith("us_") else _global_sites
        df_long = _insert_nonchannel_dummy(df_long, tb_key, master, _site_meta_map)
        out_path = EXPORTS_DIR / f"{tb_key}_stacked_separate.csv"
        df_long.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.6f")
        ok_list.append(tb_key)
        all_frames.append(df_long)

# ── channel 후처리 (루프 종료 후) ────────────────────────────────
if channel_frames:
    print(f"\n▶ channel 후처리: {list(channel_frames.keys())}")

    for tb_key, df_long in channel_frames.items():

        # value 컬럼 dtype 보정 (int64 등 비문자열 타입 대비)
        if "value" in df_long.columns:
            df_long["value"] = df_long["value"].astype(object)

        # 1. US value 치환
        is_us = df_long["Site_Code"].str.strip().str.lower() == "us"
        df_long.loc[is_us, "value"] = (
            df_long.loc[is_us, "value"].map(us_mapping).fillna(df_long.loc[is_us, "value"])
        )

        # 2. dummy 0 삽입 (마스터 site_code × 글로벌 채널 기준)
        #    US도 global_paid_mapping 전체 채널 기준으로 더미 생성
        #    (US는 실제 데이터가 없는 글로벌 채널도 0행이 있어야 함)
        master = _us_sites if tb_key.startswith("us_") else _global_sites
        all_sites = list(master)

        _channel_master = set(global_paid_mapping.keys())
        all_values = list(set(df_long["value"].dropna().unique()) | _channel_master)

        metric_cols = df_long["metric_col"].unique()
        # trailing "_" 있는 것: 채널을 크로스곱에 포함
        # trailing "_" 없는 것: 채널 이미 포함된 컬럼 → site 기준으로만 dummy
        mc_needs_ch = [mc for mc in metric_cols if str(mc).endswith("_")]
        mc_has_ch   = [mc for mc in metric_cols if not str(mc).endswith("_")]

        meta_cols = [c for c in ["Subsidiary", "Country", "RSID"] if c in df_long.columns]
        meta_df = (
            df_long.groupby("Site_Code")[meta_cols]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
            .reset_index()
        )
        date_cols = [c for c in ["Start_Date", "End_Date", "itemId"] if c in df_long.columns]
        date_meta = (
            df_long.groupby("Site_Code")[date_cols]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
            .reset_index()
        )
        file_date_mode = {
            c: df_long[c].mode().iloc[0] if not df_long[c].mode().empty else None
            for c in date_cols
        }

        # 크로스 곱 생성
        cross_parts = []
        if mc_needs_ch and all_values:
            cross_parts.append(
                pd.MultiIndex.from_product(
                    [all_sites, all_values, mc_needs_ch],
                    names=["Site_Code", "value", "metric_col"]
                ).to_frame(index=False)
            )
        if mc_has_ch:
            # 채널 포함 컬럼: site × metric_col만 크로스 (value 차원 없음)
            _no_ch = pd.MultiIndex.from_product(
                [all_sites, mc_has_ch],
                names=["Site_Code", "metric_col"]
            ).to_frame(index=False)
            _no_ch["value"] = None
            cross_parts.append(_no_ch)

        if cross_parts:
            cross = pd.concat(cross_parts, ignore_index=True)

            existing = df_long[["Site_Code", "value", "metric_col"]].drop_duplicates()
            existing["_exists"] = True

            merged = cross.merge(existing, on=["Site_Code", "value", "metric_col"], how="left")
            dummy_rows = merged[merged["_exists"].isna()].drop(columns=["_exists"])

            if not dummy_rows.empty:
                dummy_rows = dummy_rows.merge(meta_df, on="Site_Code", how="left")
                dummy_rows = dummy_rows.merge(date_meta, on="Site_Code", how="left")
                for _col in meta_cols:
                    if _col not in dummy_rows.columns:
                        continue
                    _na_mask = dummy_rows[_col].isna()
                    if _na_mask.any():
                        dummy_rows.loc[_na_mask, _col] = dummy_rows.loc[_na_mask, "Site_Code"].map(
                            lambda s: _site_meta_map.get(s, {}).get(_col)
                        )
                for _col, _val in file_date_mode.items():
                    if _col in dummy_rows.columns:
                        dummy_rows[_col] = dummy_rows[_col].fillna(_val)
                dummy_rows["metric_value_origin"] = 0.0
                dummy_rows["metric_value_adj"]    = 0.0
                dummy_rows = _apply_j_cols(dummy_rows)
                df_long = pd.concat([df_long, dummy_rows], ignore_index=True)
                print(f"  {tb_key} → dummy {len(dummy_rows)}행 삽입")

        # 2-후처리. "None" 채널명 → "Other" 로 통일
        #   더미 생성 후 처리하여 metric_col에 "None" 대신 "Other"가 붙도록 함
        #   (step 4 null 체크가 채널명 "None"을 잘못 덮어쓰는 버그도 방지)
        _none_mask = df_long["value"].astype(str) == "None"
        if _none_mask.any():
            df_long.loc[_none_mask, "value"] = "Other"

        # 2-후처리2. mc_needs_ch 행 중 value=NaN인 행 제외
        #   원본 CSV에서 채널명이 NaN으로 추출된 경우 KEY 생성이 불가하므로 완전 제외
        _mc_needs_ch_mask = df_long["metric_col"].astype(str).str.endswith("_")
        _val_missing_mask = df_long["value"].isna()
        _drop_mask = _mc_needs_ch_mask & _val_missing_mask
        if _drop_mask.any():
            print(f"  {tb_key} → 채널명 NaN 행 {_drop_mask.sum()}개 제외: "
                  f"{df_long.loc[_drop_mask, 'Site_Code'].value_counts().to_dict()}")
            df_long = df_long[~_drop_mask].reset_index(drop=True)

        # 3. metric_col에 채널명 합치기 (mc_needs_ch만 value 붙임)
        _has_year    = df_long["value"].astype(str).str.contains(r'\b20\d{2}\b', regex=True, na=False)
        _mc_needs_ch = df_long["metric_col"].astype(str).str.endswith("_")
        _mc_has_ch_rows = ~_mc_needs_ch & ~_has_year   # mc_has_ch (채널 내장 컬럼)

        df_long["metric_col"] = df_long["metric_col"].astype(str).where(
            _has_year | ~_mc_needs_ch,
            df_long["metric_col"].astype(str) + df_long["value"].astype(str)
        )

        # 4. value(ITEM) 보정
        # ① _has_year 행: value → metric_name
        df_long.loc[_has_year, "value"] = df_long.loc[_has_year, "metric_name"].str.rstrip("_")
        # ② mc_has_ch 더미 행 (value=None): metric_col에서 _null_ 이후 채널명 추출
        #    원본 데이터 mc_has_ch 행은 원본 채널(value) 그대로 유지
        _value_null = df_long["value"].isna() | (df_long["value"].astype(str) == "None")
        _mc_has_ch_null = _mc_has_ch_rows & _value_null
        if _mc_has_ch_null.any():
            _emb = df_long.loc[_mc_has_ch_null, "metric_col"].astype(str).str.extract(
                r'_null_(.+)$', expand=False
            )
            _valid = _emb.notna()
            if _valid.any():
                df_long.loc[_emb[_valid].index, "value"] = _emb[_valid].values
        # ③ 그래도 None/NaN: metric_name에서 추출
        _value_null2 = df_long["value"].isna() | (df_long["value"].astype(str) == "None")
        df_long.loc[_value_null2, "value"] = df_long.loc[_value_null2, "metric_name"].str.rstrip("_")

        # 5. PAID_NONPAID 부여
        #    - mc_needs_ch 행: value(채널명) 기준
        #    - mc_has_ch 행: metric_col에 내장된 채널 기준 (원본 row 채널과 무관하게 일관성 보장)
        _ch_for_paid = df_long["value"].copy()
        if _mc_has_ch_rows.any():
            _emb_paid = df_long.loc[_mc_has_ch_rows, "metric_col"].astype(str).str.extract(
                r'_null_(.+)$', expand=False
            )
            _valid_paid = _emb_paid.notna()
            if _valid_paid.any():
                _ch_for_paid.loc[_emb_paid[_valid_paid].index] = _emb_paid[_valid_paid].values

        df_long["PAID_NONPAID"] = _ch_for_paid.map(global_paid_mapping)
        val_idx = df_long.columns.get_loc("value")
        cols = list(df_long.columns)
        cols.remove("PAID_NONPAID")
        cols.insert(val_idx, "PAID_NONPAID")
        df_long = df_long[cols]

        out_path = EXPORTS_DIR / f"{tb_key}_stacked_separate.csv"
        finalize_df(df_long).to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.6f")
        ok_list.append(tb_key)
        if _REPORT_NO_PAT.search(tb_key):
            all_frames.append(df_long)

# ── union 생성 ────────────────────────────────────────────────────
if all_frames:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    union_path = EXPORTS_DIR / f"union_{ts}.csv"
    union_df = pd.concat(all_frames, ignore_index=True)
    if "PAID_NONPAID" in union_df.columns:
        union_df["PAID_NONPAID"] = union_df["PAID_NONPAID"].fillna("-")

    # ── US dummy 보완: non-US에 있고 US에 없는 모든 metric_col에 0행 삽입
    #    (non-channel: _cmp_, _all_ 등 / channel: 각 채널명 포함 key 등)
    _non_us_mask = union_df["Site_Code"].str.strip().str.upper() != "US"
    _us_mask     = union_df["Site_Code"].str.strip().str.upper() == "US"
    _non_us_mc   = set(union_df.loc[_non_us_mask, "metric_col"].dropna().unique())
    _us_mc       = set(union_df.loc[_us_mask,     "metric_col"].dropna().unique())
    _missing_for_us = _non_us_mc - _us_mc

    if _missing_for_us:
        _us_rows = union_df[_us_mask]
        if _us_rows.empty:
            print("[WARN] No US data → skip US dummy insert")
        else:
            _tmpl = _us_rows.iloc[0].to_dict()
            _dummy_list = []
            for _mc in sorted(_missing_for_us, key=str):
                _ref = union_df.loc[_non_us_mask & (union_df["metric_col"] == _mc)]
                _row = {col: None for col in union_df.columns}
                _row.update(_tmpl)
                _row["metric_col"] = _mc
                _row["metric_value_origin"] = 0.0
                _row["metric_value_adj"]    = 0.0
                if not _ref.empty:
                    for _dc in ["Start_Date", "End_Date"]:
                        if _dc in _ref.columns:
                            _row[_dc] = _ref.iloc[0][_dc]
                _j = split_metric_col(_mc)
                _row.update(_j)
                _row["value"] = _j.get("metric_name", "")
                if "PAID_NONPAID" in union_df.columns:
                    _row["PAID_NONPAID"] = union_df.loc[
                        _non_us_mask & (union_df["metric_col"] == _mc), "PAID_NONPAID"
                    ].mode().iloc[0] if not union_df.loc[
                        _non_us_mask & (union_df["metric_col"] == _mc), "PAID_NONPAID"
                    ].mode().empty else "-"
                _dummy_list.append(_row)
            _us_dummy_df = pd.DataFrame(_dummy_list, columns=union_df.columns)
            union_df = pd.concat([union_df, _us_dummy_df], ignore_index=True)
            print(f"US 누락 metric_col dummy {len(_dummy_list)}개 삽입")

    finalize_df(union_df).to_csv(union_path, index=False, encoding="utf-8-sig", float_format="%.6f")
    print(f"\n▶ union 저장: {union_path} ({len(union_df):,}행)")

# ── 결과 요약 ─────────────────────────────────────────────────────
print(f"\n✅ 완료: {len(ok_list)}개")
for t in ok_list: print(f"  - {t}")
print(f"\n⏭️  리포트 번호 없어서 skip: {len(no_report_list)}개")
for t in no_report_list: print(f"  - {t}")
print(f"\n⚠️  매핑 없어서 skip: {len(skip_list)}개")
for t in skip_list: print(f"  - {t}")
print(f"\n⚠️  매핑컬럼 불일치 skip: {len(no_vars_list)}개")
for t in no_vars_list: print(f"  - {t}")